# Практика · Тема 14 · Арифметика векторів

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)

Тут ми перевіряємо на власному корпусі найвідомішу демонстрацію ембедингів —
**аналогію** «король − чоловік + жінка = королева». Питання просте: чи вона працює
на 675 тисячах слововживань українських перекладів інтерфейсів?

Що зробимо:

1. зберемо корпус і навчимо skip-gram із негативним семплюванням — три зерна;
2. подивимось на **геометрію**: чому близькість міряють косинусом, а не відстанню;
3. **автоматично** побудуємо набір аналогій із морфології (`pymorphy3`), а не з голови;
4. заміряємо точність аналогій і **пастку виключення** слів запиту;
5. порівняємо арифметику з тупою базою «найближчий до `c`»;
6. перевіримо, чи це не просто недонавчена модель;
7. перевіримо, чи не заважає аналогіям спільний напрямок усіх векторів;
8. намалюємо `TSNE` і **заміряємо, наскільки картинка бреше**.

> ⏱ Зошит навчає три моделі skip-gram по пʼять епох кожна. Заміряно процесорним
> годинником на чотирьох ядрах без відеокарти: **дві — три хвилини**
> процесорного часу, з них на навчання йде близько 110-140 секунд. Останньою
> клітинкою зошит друкує цей час, тож обіцянку можна перевірити.
> На завантаженій машині стінний час буде помітно більший — до шести хвилин.

## 0 · Фіксуємо потоки **до** імпорту numpy

Це не косметика. Без цих трьох рядків бібліотеки лінійної алгебри запускають
стільки потоків, скільки в машині ядер, і ті потоки крутяться в очікуванні одне
одного. Очікування рахується як робота, і `time.process_time()` починає показувати
час, більший за справжній у десятки разів.

Рядки мусять стояти **перед** `import numpy`, бо бібліотека читає ці змінні один
раз при завантаженні.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys, re, glob, gettext, time, collections
import numpy as np
import scipy.sparse as sp

print('Python     ', sys.version.split()[0])
print('numpy      ', np.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)
import pymorphy3; print('pymorphy3  ', pymorphy3.__version__)
print('ядер у машині:', os.cpu_count(), '· потоків дозволено: 1')

## 1 · Корпус

Той самий, що в усьому курсі: українські переклади інтерфейсів із системних
файлів `.mo`. Токенізатор — канонічний для курсу, той самий шаблон, що в темах
04, 05 і 09. Якщо української локалі на машині немає, вмикається вбудований
мінікорпус — тоді числа будуть інші, і зошит про це скаже прямо.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"   # апостроф — звʼязка всередині слова
token_re = re.compile(TOKEN_PATTERN)

# запасний мінікорпус: якщо локалі немає, зошит однаково має виконатись
FALLBACK = [
    'Не вдалося відкрити файл конфігурації для читання',
    'Помилка читання файла: доступ заборонено системою',
    'Файли не знайдено у вказаному каталозі призначення',
] * 400

def load_corpus():
    """Повертає список перекладених рядків. Друкує, який шлях спрацював."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
            for src, dst in catalog._catalog.items():
                if isinstance(src, str) and isinstance(dst, str) \
                   and len(dst) > 30 and 'Project-Id' not in dst:
                    docs.append(dst)
        except Exception:
            pass
    if len(docs) < 1000:
        print('⚠ української локалі не знайдено — беремо вбудований мінікорпус')
        return FALLBACK, False
    return docs, True

documents, real_corpus = load_corpus()
token_docs = [token_re.findall(d.lower()) for d in documents]
word_counts = collections.Counter()
for t in token_docs:
    word_counts.update(t)

MIN_COUNT = 10
vocab = [w for w, c in word_counts.most_common() if c >= MIN_COUNT]
word_to_id = {w: i for i, w in enumerate(vocab)}
total_tokens = sum(word_counts.values())
kept_tokens = sum(word_counts[w] for w in vocab)

print('документів   ', len(documents))
print('слововживань ', total_tokens)
print('словоформ    ', len(word_counts))
print('словник при min_count=%d: %d слів' % (MIN_COUNT, len(vocab)))
print('покриття словником: %.4f' % (kept_tokens / total_tokens))

## 2 · Пари «слово — контекст»

Skip-gram учиться на парах: для кожного слова беремо всі слова у вікні ±5 навколо
нього. Документи в нас короткі, тож вікно майже ніколи не заповнюється повністю —
але пар усе одно набирається кілька мільйонів.

In [ ]:
WINDOW = 5

def make_pairs(token_docs, word_to_id, window=WINDOW):
    """Усі пари (центральне слово, слово з вікна) як два масиви індексів."""
    centers, contexts = [], []
    for tokens in token_docs:
        ids = [word_to_id[w] for w in tokens if w in word_to_id]
        n = len(ids)
        for i in range(n):
            lo = max(0, i - window)
            hi = min(n, i + window + 1)
            for j in range(lo, hi):
                if j != i:
                    centers.append(ids[i])
                    contexts.append(ids[j])
    return np.array(centers, dtype=np.int32), np.array(contexts, dtype=np.int32)

t0 = time.process_time()
centers, contexts = make_pairs(token_docs, word_to_id)
print('пар (слово, контекст) при вікні ±%d: %d' % (WINDOW, len(centers)))
print('на одне слововживання припадає пар: %.4f' % (len(centers) / kept_tokens))
print('побудова пар: %.1f с процесорних' % (time.process_time() - t0))

## 3 · Skip-gram із негативним семплюванням

`gensim` у середовищі немає, тож пишемо самі — і це навіть краще, бо формула
лишається видимою.

Одна деталь потребує пояснення. У ході навчання нам треба додавати градієнти до
рядків матриці за списком індексів, у якому **індекси повторюються**. Звичайне
`W[rows] += values` мовчки втрачає повтори: воно записує останнє значення, а не
суму. Правильний спосіб — `np.add.at`, але він дуже повільний. Ми беремо третій:
множення на розріджену матрицю-«роздавач», яке робить рівно те саме, але швидше.
Нижче стоїть `assert`, який це доводить.

In [ ]:
def scatter_add(matrix, rows, values):
    """matrix[rows] += values, з правильним підсумовуванням повторів індексів."""
    n = len(rows)
    picker = sp.csr_matrix(
        (np.ones(n, dtype=np.float32), rows.astype(np.int32),
         np.arange(n + 1, dtype=np.int32)),
        shape=(n, matrix.shape[0]))
    matrix += picker.T @ values

# перевірка: наш швидкий спосіб = повільний еталонний np.add.at
check_rng = np.random.default_rng(0)
rows_demo = check_rng.integers(0, 7, size=40).astype(np.int32)
vals_demo = check_rng.random((40, 3)).astype(np.float32)
slow = np.zeros((7, 3), dtype=np.float32); np.add.at(slow, rows_demo, vals_demo)
fast = np.zeros((7, 3), dtype=np.float32); scatter_add(fast, rows_demo, vals_demo)
assert np.allclose(slow, fast, atol=1e-5), 'розрахунок розійшовся!'
print('✅ scatter_add збігається з np.add.at')
print('   повторів індексів у прикладі:', 40 - len(set(rows_demo.tolist())))

Тепер саме навчання. Для кожної пари (`center`, `context`) модель штовхає їхні
вектори одне до одного, а заразом відштовхує `center` від пʼяти випадкових слів,
узятих із частотного розподілу в степені 0.75. Це і є «негативне семплювання».

Ми зберігаємо **знімки** ваг після 1, 2, 3 і 5 епохи — знадобляться, коли
перевірятимемо, чи не занадто мало ми вчили.

In [ ]:
DIM, NEG, EPOCHS, LR, BATCH = 64, 5, 5, 0.025, 2048

def train_skipgram(centers, contexts, vocab_size, counts, dim=DIM, neg=NEG,
                   epochs=EPOCHS, lr=LR, seed=0, batch=BATCH, snapshots=()):
    rng = np.random.default_rng(seed)
    W_in = ((rng.random((vocab_size, dim)) - 0.5) / dim).astype(np.float32)
    W_out = np.zeros((vocab_size, dim), dtype=np.float32)

    # негативні приклади тягнемо з частоти в степені 0.75: рідкісним словам
    # так дістається більше уваги, ніж дала б чиста частота
    probs = counts.astype(np.float64) ** 0.75
    probs /= probs.sum()
    neg_table = rng.choice(vocab_size, size=2_000_000, p=probs).astype(np.int32)

    n = len(centers)
    total_steps = epochs * ((n + batch - 1) // batch)
    step = 0
    saved = {}
    for epoch in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch):
            idx = order[start:start + batch]
            c, o = centers[idx], contexts[idx]
            negs = neg_table[rng.integers(0, len(neg_table), size=(len(idx), neg))]
            # швидкість навчання спадає лінійно — класика word2vec
            cur_lr = np.float32(lr * max(1e-4, 1.0 - step / total_steps))
            step += 1

            v = W_in[c]                       # вектори центральних слів
            u_pos = W_out[o]                  # вектори справжніх контекстів
            u_neg = W_out[negs]               # вектори випадкових слів

            score_pos = np.einsum('bd,bd->b', v, u_pos)
            score_neg = np.einsum('bd,bkd->bk', v, u_neg)
            # похідна логістичної втрати: для справжньої пари ціль 1, для випадкової 0
            g_pos = (1.0 / (1.0 + np.exp(-score_pos)) - 1.0).astype(np.float32)
            g_neg = (1.0 / (1.0 + np.exp(-score_neg))).astype(np.float32)

            grad_v = g_pos[:, None] * u_pos + np.einsum('bk,bkd->bd', g_neg, u_neg)
            scatter_add(W_out, o, (-cur_lr * g_pos)[:, None] * v)
            scatter_add(W_out, negs.ravel(),
                        ((-cur_lr * g_neg)[:, :, None] * v[:, None, :]).reshape(-1, dim))
            scatter_add(W_in, c, -cur_lr * grad_v)
        if (epoch + 1) in snapshots:
            saved[epoch + 1] = W_in.copy()
    return W_in, W_out, saved

counts = np.array([word_counts[w] for w in vocab])
# W_in — «центральна» матриця, її й беруть за вектори слів; W_out — контекстна,
# її зазвичай викидають. У розділі 12 ми перевіримо, чи справедливо
models, out_models, snapshots_by_seed, train_times = {}, {}, {}, []
for seed in (0, 1, 2):
    t0 = time.process_time()
    W, W_ctx, saved = train_skipgram(centers, contexts, len(vocab), counts,
                                     seed=seed, snapshots=(1, 2, 3, 5))
    dt = time.process_time() - t0
    train_times.append(dt)
    models[seed] = W
    out_models[seed] = W_ctx
    snapshots_by_seed[seed] = saved
    print('зерно %d · %d епох · %d вимірів · %.1f с процесорних' % (seed, EPOCHS, DIM, dt))
print('разом на навчання: %.1f с процесорних' % sum(train_times))

Перш ніж щось міряти — переконаймось, що модель узагалі вивчилась. Найдешевша
перевірка: подивитись на найближчих сусідів кількох слів. Якщо там сміття, далі
можна не йти.

In [ ]:
def unit_rows(matrix):
    """Кожен рядок ділимо на його довжину — далі скалярний добуток дорівнює косинусу."""
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)

V = unit_rows(models[0])

def nearest(word, k=5, vectors=None):
    vectors = V if vectors is None else vectors
    i = word_to_id[word]
    sims = vectors @ vectors[i]
    sims[i] = -9.0                     # саме слово не рахуємо власним сусідом
    return [vocab[j] for j in np.argsort(-sims)[:k]]

for word in ['файл', 'помилка', 'вікно', 'мережі', 'пароль']:
    if word in word_to_id:
        print('%-9s → %s' % (word, ' · '.join(nearest(word))))

## 4 · Геометрія: чому косинус, а не відстань

Тут ховається перше, що варто зрозуміти про аналогії. Вектор слова має **напрямок**
і **довжину**, і довжина не про зміст — вона про частоту. Часті слова модель
оновлює частіше, і їхні вектори виростають довшими.

Порахуймо це прямо: розібʼємо словник на смуги за частотним рангом і подивимось
середню довжину вектора в кожній.

In [ ]:
norms = np.linalg.norm(models[0], axis=1)
by_freq = np.argsort(-counts)
bands = [(0, 100), (100, 500), (500, 1500), (1500, 3000), (3000, len(vocab))]
print('%-14s %10s %12s' % ('ранг у словнику', 'довжина |v|', 'частота'))
for lo, hi in bands:
    idx = by_freq[lo:hi]
    print('%6d-%-7d %10.4f %12.1f' % (lo, hi, norms[idx].mean(), counts[idx].mean()))
corr = np.corrcoef(np.log(counts), norms)[0, 1]
print('\nкореляція log(частота) з довжиною вектора: %.4f' % corr)

### Числа цієї фігури — для лекції

Щоб інтерактив лекції малював **справжні** точки, а не намальовані, друкуємо
координати: частоту слова й довжину його вектора. Беремо кожне сороковe слово
словника, щоб покрити весь діапазон частот.

In [ ]:
step = max(1, len(vocab) // 120)
sample_ids = by_freq[::step]
print('точки (частота, довжина вектора) — %d слів:' % len(sample_ids))
print('[' + ','.join('[%d,%.2f]' % (counts[i], norms[i]) for i in sample_ids) + ']')

Кореляція висока — отже, якщо міряти близькість **евклідовою відстанню**, часті
слова опиняться «далеко від усіх» просто тому, що вони довгі. Косинус дивиться
лише на кут і цієї вади не має.

Перевірмо на числах: наскільки часто найближчий сусід за косинусом і найближчий
за евклідовою відстанню — це різні слова?

In [ ]:
def nearest_id_cosine(i, matrix_unit):
    sims = matrix_unit @ matrix_unit[i]
    sims[i] = -9.0
    return int(np.argmax(sims))

def nearest_id_euclid(i, matrix_raw):
    diff = matrix_raw - matrix_raw[i]
    dist = np.einsum('nd,nd->n', diff, diff)
    dist[i] = 9e18
    return int(np.argmin(dist))

raw = models[0]
probe = by_freq[:800]                      # 800 найчастіших слів
same = 0
for i in probe:
    if nearest_id_cosine(int(i), V) == nearest_id_euclid(int(i), raw):
        same += 1
print('на %d словах перший сусід збігся у %d випадках — %.4f' %
      (len(probe), same, same / len(probe)))

# а якщо спершу нормувати вектори, обидві міри дають те саме слово завжди
same_after = 0
for i in probe:
    if nearest_id_cosine(int(i), V) == nearest_id_euclid(int(i), V):
        same_after += 1
print('після нормування вектори збігаються у %d випадках — %.4f' %
      (same_after, same_after / len(probe)))

Друге число рівне одиниці, і це не випадковість, а алгебра. Для векторів однакової
довжини квадрат відстані дорівнює `2 − 2·cos`: чим більший косинус, тим менша
відстань, і порядок сусідів у двох мір **тотожний**. Тобто «косинус проти
евклідової відстані» — насправді питання про **нормування**, а не про формулу.

## 5 · Набір аналогій, зібраний автоматично

Класичні набори аналогій («Афіни : Греція = Париж : Франція») до нашого корпусу не
підходять: у перекладах інтерфейсів немає ані королів, ані столиць. Але в нас є те,
чого немає в англійських наборах, — **багата морфологія**. `файл : файли` і
`помилка : помилки` — це готова аналогія за числом, і зібрати такі пари можна
машиною, а не з голови.

Беремо кожне слово словника, розбираємо `pymorphy3`, просимо змінити рівно одну
граматичну ознаку й лишаємо пару, якщо обидві форми є у словнику.

⚠ Одна пастка українського словника `pymorphy3`: **однина не має власної позначки**.
У розборі «файл» стоїть `NOUN,inan masc,nomn` — слова `sing` там немає. Тому однину
задаємо від протилежного: «серед ознак немає `plur`».

In [ ]:
morph = pymorphy3.MorphAnalyzer(lang='uk')

# (назва, частина мови, потрібні ознаки, заборонені ознаки, цільові ознаки)
RELATIONS = [
    ('число іменника',              'NOUN', {'nomn'},         {'plur'}, {'plur', 'nomn'}),
    ('відмінок іменника',           'NOUN', {'nomn'},         {'plur'}, {'gent'}),
    ('рід прикметника',             'ADJF', {'masc', 'nomn'}, set(),    {'femn', 'nomn'}),
    ('число прикметника',           'ADJF', {'masc', 'nomn'}, set(),    {'plur', 'nomn'}),
    ('дієслово → 3 особа',          'VERB', {'infn'},         set(),    {'3per'}),
    ('дієслово → безособова форма', 'VERB', {'infn'},         set(),    {'Impe'}),
    ('дієслово → минулий час',      'VERB', {'infn'},         set(),    {'past', 'masc'}),
]

def build_relation_pairs(vocab, relations):
    vocab_set = set(vocab)
    result = {}
    for name, pos, need, forbid, target in relations:
        pairs = []
        for word in vocab:
            for parse in morph.parse(word):
                grammemes = set(parse.tag.grammemes)
                if parse.tag.POS != pos:
                    continue
                if not need <= grammemes or (forbid & grammemes):
                    continue
                try:
                    other = parse.inflect(target)
                except ValueError:
                    other = None
                if other is None or other.word == word or other.word not in vocab_set:
                    continue
                pairs.append((word, other.word))
                break        # беремо лише перший розбір, щоб не задвоювати слово
        result[name] = sorted(set(pairs))
    return result

t0 = time.process_time()
relation_pairs = build_relation_pairs(vocab, RELATIONS)
print('розбір словника: %.1f с процесорних\n' % (time.process_time() - t0))
print('%-30s %5s   приклади' % ('відношення', 'пар'))
for name, pairs in relation_pairs.items():
    sample = ', '.join('%s→%s' % p for p in pairs[:3])
    print('%-30s %5d   %s' % (name, len(pairs), sample))
print('\nусього пар:', sum(len(p) for p in relation_pairs.values()))

### Перша неприємність, і вона суто українська

Подивись на пари «число іменника» й «відмінок іменника». Для іменників жіночого
роду форма множини називного відмінка і форма однини родового **збігаються
буквою**: `помилки` — це і «кілька помилок», і «немає помилки». Модель бачить один
рядок символів і один вектор; розрізнити два значення їй нічим.

Скільки пар потрапило в обидва набори одночасно?

In [ ]:
set_number = set(relation_pairs['число іменника'])
set_case = set(relation_pairs['відмінок іменника'])
both = set_number & set_case
print('пар «число іменника»   :', len(set_number))
print('пар «відмінок іменника»:', len(set_case))
print('однакових пар в обох   :', len(both))
print('це %.4f усіх пар «число іменника»' % (len(both) / len(set_number)))
print('\nприклади збігу:', ', '.join('%s→%s' % p for p in sorted(both)[:5]))

## 6 · Як влаштований замір аналогії

З пар одного відношення робимо четвірки: якщо `a → b` і `c → d` — це одне й те саме
граматичне перетворення, то має виконуватись `b − a + c ≈ d`.

Далі рахуємо косинус між `b − a + c` і **кожним** словом словника й дивимось, яке
виявиться першим. Класична реалізація при цьому **викидає з відповіді самі слова
запиту** — `a`, `b`, `c`. Ми рахуємо обидва варіанти, з викиданням і без, бо різниця
між ними і є головним сюжетом теми.

In [ ]:
def make_analogies(pairs, n_max=1500, seed=0):
    """З набору пар одного відношення робимо четвірки a:b = c:d."""
    rng = np.random.default_rng(seed)
    n = len(pairs)
    combos = [(i, j) for i in range(n) for j in range(n) if i != j]
    if len(combos) > n_max:
        picked = rng.choice(len(combos), size=n_max, replace=False)
        combos = [combos[k] for k in sorted(picked)]
    quads = []
    for i, j in combos:
        a, b = pairs[i]
        c, d = pairs[j]
        if len({a, b, c, d}) < 4:        # вироджені четвірки нічого не перевіряють
            continue
        quads.append((a, b, c, d))
    return quads

analogies = {}
for name, pairs in relation_pairs.items():
    if len(pairs) < 3:
        continue
    analogies[name] = make_analogies(pairs)
for name, quads in analogies.items():
    print('%-30s четвірок: %5d   напр. %s : %s = %s : %s' % (name, len(quads), *quads[0]))

In [ ]:
def evaluate_analogies(quads, vectors_unit, block=256, full=True):
    """Повертає всі заміри одразу: з викиданням, без, 3CosMul, евклід, «найближчий до c»."""
    A = np.array([word_to_id[q[0]] for q in quads])
    B = np.array([word_to_id[q[1]] for q in quads])
    C = np.array([word_to_id[q[2]] for q in quads])
    D = np.array([word_to_id[q[3]] for q in quads])
    n = len(quads)
    pred = {k: np.zeros(n, dtype=np.int32)
            for k in ('add_excl', 'add_raw', 'mul', 'euclid', 'near_c')}
    rank_of_d = np.zeros(n, dtype=np.int32)

    for s in range(0, n, block):
        sl = slice(s, min(n, s + block))
        a, b, c, d = A[sl], B[sl], C[sl], D[sl]
        row = np.arange(len(a))
        query = vectors_unit[b] - vectors_unit[a] + vectors_unit[c]
        sims = query @ vectors_unit.T                      # (порція, увесь словник)

        pred['add_raw'][sl] = sims.argmax(1)               # нічого не викидаємо
        blocked = sims.copy()
        blocked[row, a] = -9.0; blocked[row, b] = -9.0; blocked[row, c] = -9.0
        pred['add_excl'][sl] = blocked.argmax(1)
        rank_of_d[sl] = (blocked > blocked[row, d][:, None]).sum(1) + 1

        if not full:            # для швидких прогонів рахуємо лише головну міру
            continue

        # евклідова відстань до тієї самої точки b−a+c
        dist = 1.0 - 2.0 * sims + np.einsum('bd,bd->b', query, query)[:, None]
        dist[row, a] = 9e18; dist[row, b] = 9e18; dist[row, c] = 9e18
        pred['euclid'][sl] = dist.argmin(1)

        # 3CosMul: множимо близькості замість того, щоб їх складати
        cos_a = np.clip(vectors_unit[a] @ vectors_unit.T, -1, 1)
        cos_b = np.clip(vectors_unit[b] @ vectors_unit.T, -1, 1)
        cos_c = np.clip(vectors_unit[c] @ vectors_unit.T, -1, 1)
        mul = ((cos_b + 1) / 2) * ((cos_c + 1) / 2) / (((cos_a + 1) / 2) + 1e-3)
        mul[row, a] = -9.0; mul[row, b] = -9.0; mul[row, c] = -9.0
        pred['mul'][sl] = mul.argmax(1)

        # тупа база: жодної арифметики, просто найближчий до c
        plain = cos_c.copy()
        plain[row, a] = -9.0; plain[row, b] = -9.0; plain[row, c] = -9.0
        pred['near_c'][sl] = plain.argmax(1)

    hit = {k: (pred[k] == D) for k in pred}
    return {'n': n, 'A': A, 'B': B, 'C': C, 'D': D,
            'pred': pred, 'hit': hit, 'rank': rank_of_d}

t0 = time.process_time()
results = {}
for seed in (0, 1, 2):
    Vs = unit_rows(models[seed])
    results[seed] = {name: evaluate_analogies(quads, Vs)
                     for name, quads in analogies.items()}
print('оцінка аналогій: %.1f с процесорних' % (time.process_time() - t0))
print('замірів: %d відношень × 3 зерна' % len(analogies))

## 7 · Головне число теми

Точність — частка четвірок, де перше слово після `b − a + c` виявилось саме `d`.
Три зерна дають розкид; різниця, менша за розкид, різницею не є.

In [ ]:
def mean_spread(name, key):
    vals = [results[s][name]['hit'][key].mean() for s in (0, 1, 2)]
    return float(np.mean(vals)), float(np.std(vals)), vals

print('%-30s %5s  %-16s %-16s' % ('відношення', 'N', '3CosAdd', 'без викидання'))
for name in analogies:
    m1, s1, _ = mean_spread(name, 'add_excl')
    m2, s2, _ = mean_spread(name, 'add_raw')
    print('%-30s %5d  %.4f ±%.4f  %.4f ±%.4f' %
          (name, results[0][name]['n'], m1, s1, m2, s2))

# зважене за кількістю четвірок середнє по всіх відношеннях
weights = np.array([results[0][n]['n'] for n in analogies], dtype=float)
all_hits = {s: np.concatenate([results[s][n]['hit']['add_excl'] for n in analogies])
            for s in (0, 1, 2)}
acc_all = [all_hits[s].mean() for s in (0, 1, 2)]
print('\nПО ВСІХ ВІДНОШЕННЯХ: %.4f ±%.4f  (по зернах %s)' %
      (np.mean(acc_all), np.std(acc_all), ' · '.join('%.4f' % a for a in acc_all)))
print('усього четвірок:', int(weights.sum()))
print('випадкове вгадування дало б: %.6f' % (1.0 / len(vocab)))
print('тобто ми в %.1f раза кращі за випадковість — і все одно помиляємось у %.1f %% випадків'
      % (np.mean(acc_all) * len(vocab), 100 * (1 - np.mean(acc_all))))

Точність у кілька відсотків — це **поразка**, і її треба назвати поразкою. Але
«помилились» не означає «нічого не знають»: подивімось, на якому місці стоїть
правильна відповідь, коли вона не перша.

In [ ]:
print('%-30s %8s %8s %8s %10s' % ('відношення', 'hit@1', 'hit@5', 'hit@10', 'медіана рангу'))
for name in analogies:
    ranks = np.concatenate([results[s][name]['rank'] for s in (0, 1, 2)])
    print('%-30s %8.4f %8.4f %8.4f %10d' %
          (name, (ranks == 1).mean(), (ranks <= 5).mean(),
           (ranks <= 10).mean(), int(np.median(ranks))))
all_ranks = np.concatenate([results[s][n]['rank'] for s in (0, 1, 2) for n in analogies])
print('\nпо всіх: hit@1 %.4f · hit@5 %.4f · hit@10 %.4f · медіана рангу %d з %d' %
      ((all_ranks == 1).mean(), (all_ranks <= 5).mean(), (all_ranks <= 10).mean(),
       int(np.median(all_ranks)), len(vocab)))

## 8 · Пастка виключення

А тепер про те, за що аналогії критикують уже понад десять років.

Класична реалізація **викидає з відповіді слова самого запиту**. Мотив нібито
безневинний: «ми ж не хочемо, щоб на запит `файл : файли = помилка : ?` модель
відповіла `помилка`». Але це виключення тихо перетворює важку задачу на легшу — і
частина «правильних» відповідей існує лише завдяки йому.

Порахуймо, скільки: у скількох четвірках модель **без** виключення відповіла б `a`,
`b` або `c` — і при цьому **з** виключенням дала правильне `d`.

In [ ]:
print('%-30s %9s %9s %9s %11s' %
      ('відношення', 'дає a', 'дає b', 'дає c', 'разом вхідне'))
for name in analogies:
    parts = []
    for key in ('A', 'B', 'C'):
        share = np.mean([(results[s][name]['pred']['add_raw'] == results[s][name][key]).mean()
                         for s in (0, 1, 2)])
        parts.append(share)
    raw_is_input = np.mean([
        ((results[s][name]['pred']['add_raw'] == results[s][name]['A']) |
         (results[s][name]['pred']['add_raw'] == results[s][name]['B']) |
         (results[s][name]['pred']['add_raw'] == results[s][name]['C'])).mean()
        for s in (0, 1, 2)])
    print('%-30s %9.4f %9.4f %9.4f %11.4f' % (name, *parts, raw_is_input))

In [ ]:
rescued_all, ok_all, input_all, raw_ok_all = [], [], [], []
for s in (0, 1, 2):
    ok = np.concatenate([results[s][n]['hit']['add_excl'] for n in analogies])
    raw = np.concatenate([results[s][n]['pred']['add_raw'] for n in analogies])
    A = np.concatenate([results[s][n]['A'] for n in analogies])
    B = np.concatenate([results[s][n]['B'] for n in analogies])
    C = np.concatenate([results[s][n]['C'] for n in analogies])
    D = np.concatenate([results[s][n]['D'] for n in analogies])
    is_input = (raw == A) | (raw == B) | (raw == C)
    rescued_all.append(int((ok & is_input).sum()))
    ok_all.append(int(ok.sum()))
    input_all.append(float(is_input.mean()))
    raw_ok_all.append(float((raw == D).mean()))

print('відповідь БЕЗ виключення — це слово самого запиту у %.4f випадків' % np.mean(input_all))
print('точність БЕЗ виключення (чиста арифметика): %.4f ±%.4f'
      % (np.mean(raw_ok_all), np.std(raw_ok_all)))
print('точність З виключенням                   : %.4f ±%.4f'
      % (np.mean(acc_all), np.std(acc_all)))
print()
print('правильних відповідей із виключенням: %.1f' % np.mean(ok_all))
print('з них існують ЛИШЕ завдяки виключенню: %.1f' % np.mean(rescued_all))
print('тобто %.4f усіх «успіхів» аналогії — заслуга не арифметики, а фільтра'
      % (np.mean(rescued_all) / np.mean(ok_all)))
print('по зернах врятовано:', rescued_all, 'із', ok_all)

## 9 · Що аналогія міряє насправді

Якщо арифметика `b − a + c` справді працює, вона має бити тупу базу, яка жодної
арифметики не робить: **просто найближче слово до `c`**. Порівняймо чотири способи
на тих самих четвірках.

In [ ]:
names = {'add_excl': '3CosAdd (b−a+c)', 'mul': '3CosMul', 'euclid': 'евклідова відстань',
         'near_c': 'найближчий до c (без арифметики)'}
print('%-34s %-18s' % ('спосіб', 'точність'))
summary = {}
for key, title in names.items():
    vals = [np.concatenate([results[s][n]['hit'][key] for n in analogies]).mean()
            for s in (0, 1, 2)]
    summary[key] = (float(np.mean(vals)), float(np.std(vals)))
    print('%-34s %.4f ±%.4f' % (title, np.mean(vals), np.std(vals)))

diff = summary['add_excl'][0] - summary['near_c'][0]
spread = max(summary['add_excl'][1], summary['near_c'][1])
print('\nарифметика мінус тупа база: %+.4f  (розкид %.4f)' % (diff, spread))
print('різниця %s за розкид' % ('БІЛЬША' if abs(diff) > spread else 'МЕНША'))

In [ ]:
print('%-30s %10s %10s %10s' % ('відношення', '3CosAdd', 'до c', 'різниця'))
for name in analogies:
    a1 = np.mean([results[s][name]['hit']['add_excl'].mean() for s in (0, 1, 2)])
    a2 = np.mean([results[s][name]['hit']['near_c'].mean() for s in (0, 1, 2)])
    print('%-30s %10.4f %10.4f %+10.4f' % (name, a1, a2, a1 - a2))

### Числа для фігур із замірами

In [ ]:
METHODS = ('add_excl', 'mul', 'euclid', 'near_c')
print('відношення | четвірок | по зернах: 3CosAdd, 3CosMul, евклід, до c | hit@5 | hit@10')
for name in analogies:
    row = [name, results[0][name]['n']]
    parts = []
    for key in METHODS:
        vals = [results[s][name]['hit'][key].mean() for s in (0, 1, 2)]
        parts.append('%s=[%s]' % (key, ','.join('%.4f' % v for v in vals)))
    ranks = np.concatenate([results[s][name]['rank'] for s in (0, 1, 2)])
    print('%-30s %5d  %s  hit5=%.4f hit10=%.4f'
          % (name, results[0][name]['n'], ' '.join(parts),
             (ranks <= 5).mean(), (ranks <= 10).mean()))

In [ ]:
print('розклад відповіді БЕЗ виключення: частка a / b / c / правильне d / інше')
for name in analogies:
    shares = []
    for key in ('A', 'B', 'C', 'D'):
        shares.append(np.mean([(results[s][name]['pred']['add_raw']
                                == results[s][name][key]).mean() for s in (0, 1, 2)]))
    other = 1.0 - sum(shares)
    print('%-30s %.4f %.4f %.4f %.4f %.4f' % (name, *shares, other))

# те саме одним рядком по всіх відношеннях — саме це число цитує лекція
totals = []
for key in ('A', 'B', 'C', 'D'):
    vals = []
    for s in (0, 1, 2):
        pred = np.concatenate([results[s][n]['pred']['add_raw'] for n in analogies])
        gold = np.concatenate([results[s][n][key] for n in analogies])
        vals.append((pred == gold).mean())
    totals.append(float(np.mean(vals)))
print('%-30s %.4f %.4f %.4f %.4f %.4f'
      % ('УСІ РАЗОМ', *totals, 1.0 - sum(totals)))

In [ ]:
print('точність із виключенням і врятовані нею четвірки, по відношеннях:')
for name in analogies:
    ok_vals, resc_vals = [], []
    for s in (0, 1, 2):
        r = results[s][name]
        ok = r['hit']['add_excl']
        is_input = ((r['pred']['add_raw'] == r['A']) | (r['pred']['add_raw'] == r['B'])
                    | (r['pred']['add_raw'] == r['C']))
        ok_vals.append(ok.mean())
        resc_vals.append((ok & is_input).mean())
    print('%-30s точність %.4f · з них завдяки фільтру %.4f (%.4f усіх успіхів)'
          % (name, np.mean(ok_vals), np.mean(resc_vals),
             np.mean(resc_vals) / max(1e-9, np.mean(ok_vals))))

### Евклідова відстань дає рівно те саме

У таблиці вище `3CosAdd` і «евклідова відстань» збіглися до останнього знака. Це не
збіг і не помилка: вектори ми пронормували, а для векторів однакової довжини
відстань — монотонна функція косинуса. Перевірмо це `assert`-ом, а не на око.

In [ ]:
same = all(np.array_equal(results[s][n]['pred']['add_excl'],
                          results[s][n]['pred']['euclid'])
           for s in (0, 1, 2) for n in analogies)
assert same, 'на нормованих векторах косинус і евклід мусять давати ТОТОЖНІ відповіді'
print('✅ на нормованих векторах косинус і евклідова відстань дають те саме слово — завжди')

# а без нормування — вже ні
raw0 = models[0]
name0 = list(analogies)[0]
quads0 = analogies[name0]
A = np.array([word_to_id[q[0]] for q in quads0]); B = np.array([word_to_id[q[1]] for q in quads0])
C = np.array([word_to_id[q[2]] for q in quads0]); D = np.array([word_to_id[q[3]] for q in quads0])
query_raw = raw0[B] - raw0[A] + raw0[C]
d2 = (np.einsum('nd,nd->n', raw0, raw0)[None, :]
      - 2 * (query_raw @ raw0.T)
      + np.einsum('bd,bd->b', query_raw, query_raw)[:, None])
row = np.arange(len(quads0))
d2[row, A] = 9e18; d2[row, B] = 9e18; d2[row, C] = 9e18
pred_raw_euclid = d2.argmin(1)
print('без нормування евклід дає точність %.4f проти %.4f у косинуса (%s)'
      % ((pred_raw_euclid == D).mean(),
         results[0][name0]['hit']['add_excl'].mean(), name0))

## 10 · А раптом ми просто мало вчили?

Це чесне заперечення, і його треба закрити числом. У нас є знімки ваг після 1, 2, 3
і 5 епохи — порахуймо точність на кожному.

In [ ]:
print('%6s  %-20s' % ('епох', 'точність по всіх відношеннях'))
curve = []
for ep in (1, 2, 3, 5):
    vals = []
    for s in (0, 1, 2):
        Vs = unit_rows(snapshots_by_seed[s][ep])
        hits = np.concatenate([evaluate_analogies(q, Vs, full=False)['hit']['add_excl']
                               for q in analogies.values()])
        vals.append(hits.mean())
    curve.append((ep, float(np.mean(vals)), float(np.std(vals))))
    print('%6d  %.4f ±%.4f' % (ep, np.mean(vals), np.std(vals)))
gain = curve[-1][1] / curve[0][1]
print('\nвід 1 до 5 епох точність зросла у %.2f раза — і все одно лишилась %.4f'
      % (gain, curve[-1][1]))

## 11 · Чому паралелограм не сходиться

Аналогія тримається на припущенні, що зсув `b − a` — **той самий вектор** для всіх
пар одного відношення. Якщо це так, чотири точки лягають у паралелограм. Перевіримо
припущення прямо: порахуємо середній косинус між зсувами всередині відношення й
порівняємо з контролем — косинусом між зсувами **різних** відношень.

In [ ]:
def offsets(pairs, vectors_unit):
    off = np.array([vectors_unit[word_to_id[b]] - vectors_unit[word_to_id[a]]
                    for a, b in pairs])
    return unit_rows(off)

V0 = unit_rows(models[0])
off_by_rel = {}
print('%-30s %10s %8s' % ('відношення', 'сталість', 'пар'))
for name, pairs in relation_pairs.items():
    if len(pairs) < 3:
        continue
    O = offsets(pairs, V0)
    off_by_rel[name] = O
    S = O @ O.T
    mask = ~np.eye(len(pairs), dtype=bool)
    print('%-30s %10.4f %8d' % (name, S[mask].mean(), len(pairs)))

cross = []
keys = list(off_by_rel)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        cross.append(float((off_by_rel[keys[i]] @ off_by_rel[keys[j]].T).mean()))
print('\nконтроль — зсуви РІЗНИХ відношень: %.4f' % np.mean(cross))
print('якби зсув був сталим вектором, у першій колонці стояла б одиниця')

## 12 · Спільний напрямок усіх векторів

Є ще одне пояснення слабкості аналогій, і воно не про метод, а про **деталь
реалізації**. Skip-gram навчає дві матриці: «центральну» `W_in` (вектор слова, коли
воно посередині) і «контекстну» `W_out` (вектор того самого слова, коли воно сусід).
За вектори слів беруть зазвичай першу, а другу викидають — так робить і Word2Vec
за замовчуванням, так робили й ми в усій темі.

Але у центральної матриці є неприємна властивість: **усі вектори трохи дивляться в
один бік**. Порахуймо середній косинус між усіма парами слів словника. Якби напрямки
були розподілені рівномірно, він був би близький до нуля.

Заміряємо три варіанти й дивимось, чи міняється від них точність аналогій:

1. центральна матриця — те, чим ми користувались;
2. сума центральної й контекстної, `v + u` — класичний прийом;
3. центральна, з якої відняли **середній вектор словника** — тобто саме той
   спільний напрямок і прибрали.

⚠ Тут легко помилитись, і я помилився з першого разу. «Прибрати першу головну
компоненту» звучить як рецепт, але головні компоненти рахуються **від середнього**,
тож найсильніший спільний напрямок вони якраз і не бачать: після такої операції
середній косинус у мене не впав, а виріс. Прибирати треба саме середнє.

In [ ]:
def mean_pairwise_cosine(matrix):
    """Середній косинус між УСІМА парами слів словника.

    Наївно це n² скалярних добутків. Але для одиничних векторів працює тотожність
    |Σu|² = n + Σ(i≠j) u_i·u_j, тож вистачає однієї суми."""
    U = unit_rows(matrix)
    s = U.sum(axis=0)
    n = len(U)
    return float((s @ s - n) / (n * (n - 1)))


def drop_common_direction(matrix):
    """Прибираємо спільний напрямок: середній вектор усього словника.

    Саме він і є причиною великого середнього косинуса. Прибрати «першу головну
    компоненту», не прибравши середнього, тут не спрацює: головні компоненти
    рахуються ВІД середнього, тож найсильніший спільний напрямок вони не бачать."""
    U = unit_rows(matrix).astype(np.float64)
    return U - U.mean(axis=0)


VARIANTS = {
    'центральна v (як усюди вище)': lambda s: models[s],
    'сума v + u': lambda s: models[s] + out_models[s],
    'центральна мінус спільний напрямок': lambda s: drop_common_direction(models[s]),
}

t0 = time.process_time()
variant_stats = {}
print('%-32s %10s %10s %10s' % ('вектори', 'сер. косинус', '3CosAdd', 'без викид.'))
for title, make in VARIANTS.items():
    iso, acc, raw = [], [], []
    for seed in (0, 1, 2):
        M = make(seed)
        iso.append(mean_pairwise_cosine(M))
        Vs = unit_rows(M)
        res = [evaluate_analogies(q, Vs, full=False) for q in analogies.values()]
        hits = np.concatenate([r['hit']['add_excl'] for r in res])
        rawhit = np.concatenate([(r['pred']['add_raw'] == r['D']) for r in res])
        acc.append(hits.mean())
        raw.append(rawhit.mean())
    variant_stats[title] = (np.mean(iso), np.std(iso), np.mean(acc), np.std(acc),
                            np.mean(raw), np.std(raw))
    print('%-32s %10.4f %10.4f %10.4f' % (title, np.mean(iso), np.mean(acc), np.mean(raw)))
print('\nте саме з розкидом по трьох зернах:')
for title, v in variant_stats.items():
    print('%-32s косинус %.4f ±%.4f · 3CosAdd %.4f ±%.4f · без викидання %.4f ±%.4f'
          % (title, v[0], v[1], v[2], v[3], v[4], v[5]))
print('\nзамір трьох варіантів: %.1f с процесорних' % (time.process_time() - t0))

Дивимось на два стовпчики разом. Якщо спільний напрямок справді заважає, то варіант
із меншим середнім косинусом мусить дати вищу точність. Порівнюємо різницю з розкидом
по трьох зернах: різниця, менша за розкид, різницею не є.

In [ ]:
base = variant_stats['центральна v (як усюди вище)']
print('%-32s %12s %12s' % ('вектори', 'Δ точності', 'розкид'))
for title, v in variant_stats.items():
    d = v[2] - base[2]
    spread = max(v[3], base[3])
    mark = 'БІЛЬША за розкид' if abs(d) > spread else 'менша за розкид'
    print('%-32s %+12.4f %12.4f   %s' % (title, d, spread, mark))
print()
print('середній косинус між усіма словами: %.4f у центральній проти %.4f у сумі v+u'
      % (base[0], variant_stats['сума v + u'][0]))

Останнє питання про спільний напрямок: **звідки він береться**? Найпростіша підозра —
що його накопичує саме навчання: чим довше вчимо, тим сильніше вектори збиваються в
конус. У нас є знімки ваг після 1, 2, 3 і 5 епохи, тож перевірити це нічого не коштує.

Забігаючи наперед: підозра не просто не підтвердилась, а **виявилась хибною із
протилежним знаком**. Дивись на напрямок стовпчика, перш ніж читати пояснення.

In [ ]:
print('%6s %14s %10s' % ('епох', 'сер. косинус', 'розкид'))
aniso_curve = []
for ep in (1, 2, 3, 5):
    vals = [mean_pairwise_cosine(snapshots_by_seed[s][ep]) for s in (0, 1, 2)]
    aniso_curve.append((ep, float(np.mean(vals)), float(np.std(vals))))
    print('%6d %14.4f %10.4f' % (ep, np.mean(vals), np.std(vals)))
print()
print('від 1 до 5 епох спільний напрямок не виріс, а ВПАВ: %.4f → %.4f, тобто у %.2f раза'
      % (aniso_curve[0][1], aniso_curve[-1][1], aniso_curve[0][1] / aniso_curve[-1][1]))
print('Підозра не підтвердилась із протилежним знаком: анізотропія — це стан, з якого')
print('навчання ВИХОДИТЬ, а не той, до якого воно приходить. Свіжі вектори майже')
print('колінеарні, і кожна епоха їх розводить.')
print()
print('Практичний наслідок: середній косинус не можна цитувати без кількості епох —')
print('на 1 епосі це %.4f, на 5 — %.4f, і це та сама модель на тому самому корпусі.'
      % (aniso_curve[0][1], aniso_curve[-1][1]))

## 13 · Візуалізація: що видно і чого не видно

`TSNE` зі `sklearn` укладає 64 виміри у два. Картинка виходить гарна — і саме тому
з нею треба поводитись обережно. Заміряймо, скільки сусідства вона зберігає.

In [ ]:
from sklearn.manifold import TSNE

TOP = 500
top_ids = by_freq[:TOP]
X = V0[top_ids]
t0 = time.process_time()
Y = TSNE(n_components=2, perplexity=30, init='pca',
         random_state=0, max_iter=500).fit_transform(X)
print('TSNE на %d словах: %.1f с процесорних' % (TOP, time.process_time() - t0))

sims_hi = X @ X.T
np.fill_diagonal(sims_hi, -9e9)
top10_hi = np.argsort(-sims_hi, axis=1)[:, :10]

dist_lo = ((Y[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
np.fill_diagonal(dist_lo, 9e18)
order_lo = np.argsort(dist_lo, axis=1)
rank_lo = np.argsort(order_lo, axis=1)

kept = np.mean([(rank_lo[i, top10_hi[i]] < 10).mean() for i in range(TOP)])
print('із топ-10 сусідів у 64 вимірах у топ-10 на картинці лишається: %.4f' % kept)

rank_hi = np.argsort(np.argsort(-sims_hi, axis=1), axis=1)
top10_lo = order_lo[:, :10]
ranks_of_shown = np.array([rank_hi[i, top10_lo[i]] for i in range(TOP)]).ravel() + 1
print('сусіди, які видно на картинці: медіанний ранг у 64 вимірах = %d з %d'
      % (int(np.median(ranks_of_shown)), TOP))
print('частка показаних «сусідів», які насправді далі за сотий ранг: %.4f'
      % (ranks_of_shown > 100).mean())

In [ ]:
print('скільки з топ-10 сусідів 64D потрапляють у топ-K на картинці:')
for K in (1, 5, 10, 25, 50, 100, 200):
    share = np.mean([(rank_lo[i, top10_hi[i]] < K).mean() for i in range(TOP)])
    print('  K = %-4d %.4f' % (K, share))

### Координати картинки TSNE

Двовимірні координати перших 220 слів — саме їх малює остання фігура лекції.
Друкуємо їх, щоб картинку можна було відтворити, а не повірити на слово.

In [ ]:
SHOW = 220
xs = Y[:SHOW, 0]; ys = Y[:SHOW, 1]
xs = (xs - xs.min()) / (xs.max() - xs.min())
ys = (ys - ys.min()) / (ys.max() - ys.min())
print('слова:', '|'.join(vocab[i] for i in top_ids[:SHOW]))
print('координати:')
print('[' + ','.join('[%.3f,%.3f]' % (x, y) for x, y in zip(xs, ys)) + ']')

In [ ]:
print('крива збереження сусідства (K, частка топ-10 сусідів 64D у топ-K на картинці):')
pairs_curve = []
for K in (1, 2, 3, 5, 8, 10, 15, 20, 25, 40, 50, 75, 100, 150, 200):
    share = np.mean([(rank_lo[i, top10_hi[i]] < K).mean() for i in range(TOP)])
    pairs_curve.append((K, share))
print('[' + ','.join('[%d,%.4f]' % (k, v) for k, v in pairs_curve) + ']')

## 14 · Ті аналогії, що все ж спрацювали

Не «гарні приклади», а повний перелік того, скільки їх. Показуємо по три з кожного
відношення — разом із тим, що модель відповіла б **без** виключення слів запиту.

In [ ]:
for name, quads in analogies.items():
    r = results[0][name]
    ok_idx = np.flatnonzero(r['hit']['add_excl'])
    print('%s — правильних %d із %d' % (name, len(ok_idx), r['n']))
    for k in ok_idx[:3]:
        a, b, c, d = quads[k]
        print('   %s : %s = %s : %s   (без виключення дало б «%s»)'
              % (a, b, c, d, vocab[r['pred']['add_raw'][k]]))
    print()

## 15 · Підсумок зошита

In [ ]:
print('корпус                       : %d слововживань, словник %d' % (total_tokens, len(vocab)))
print('пар (слово, контекст)        : %d' % len(centers))
print('модель                       : skip-gram %d вимірів, %d епох, %d негативних'
      % (DIM, EPOCHS, NEG))
print('навчання                     : %.1f с процесорних на три зерна' % sum(train_times))
print('аналогій зібрано автоматично : %d четвірок у %d відношеннях'
      % (sum(len(q) for q in analogies.values()), len(analogies)))
print()
print('ТОЧНІСТЬ АНАЛОГІЙ            : %.4f ±%.4f' % (np.mean(acc_all), np.std(acc_all)))
print('тупа база «найближчий до c»  : %.4f ±%.4f' % summary['near_c'])
print('частка «успіхів» від фільтра : %.4f' % (np.mean(rescued_all) / np.mean(ok_all)))
print('сусідство, збережене на TSNE : %.4f' % kept)
print()
print('точність БЕЗ виключення слів   : %.4f ±%.4f' % (np.mean(raw_ok_all), np.std(raw_ok_all)))
print()
print('Висновок: на корпусі такого розміру арифметика векторів не працює.')
print()
# скільки процесорного часу коштував зошит цілком — щоб обіцянка вгорі була заміряною
print('увесь зошит                  : %.0f с процесорного часу' % time.process_time())

## Завдання

**🟢 Рівень 1.** Додай до `RELATIONS` ще одне граматичне відношення (наприклад,
місцевий відмінок `loct` замість родового) і заміряй його точність трьома зернами.
*Зроблено, якщо* ти назвав кількість пар, точність із розкидом і сказав, чи вона
відрізняється від сусідніх відношень більше за розкид.

**🟡 Рівень 2.** Обмеж набір аналогій словами, що трапляються частіше за 100 разів,
і переміряй усе. *Зроблено, якщо* ти показав дві таблиці поруч і пояснив, чому
точність змінилась саме так.

**🔴 Рівень 3.** Реалізуй заміну зсуву: замість `b − a` для однієї пари візьми
**середній зсув по всьому відношенню** й шукай найближче до `c + середній_зсув`.
*Зроблено, якщо* ти порівняв три способи (`3CosAdd`, середній зсув, «найближчий
до c») на трьох зернах і сказав, який виграв і на скільки розкидів.